In [45]:
import pyEDM
import pandas as pd
import numpy as np
import random

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [46]:
env = pd.read_csv('../data/environment_all.csv')
env = env.drop(columns=['fluorescent_dissolved_organic_matter_eco', 'sea_water_turbidity_eco', 'waterlevel_predicted_m'])

# ones of contention (due to too many missing values):
    # mass_concentration_of_oxygen_in_sea_water_seaphox
    # mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox
    # fractional_saturation_of_oxygen_in_sea_water_seaphox
    # sea_water_ph_reported_on_total_scale_seaphox_external

env = env.drop(columns=['mass_concentration_of_oxygen_in_sea_water_seaphox', 'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox', 'fractional_saturation_of_oxygen_in_sea_water_seaphox', 'sea_water_ph_reported_on_total_scale_seaphox_external'])
env.head()

,date,mass_concentration_of_chlorophyll_in_sea_water_ctd,sea_water_electrical_conductivity_ctd,sea_water_practical_salinity_ctd,sea_water_sigma_t_ctd,sea_water_pressure_ctd,sea_water_temperature_ctd,wind_speed_ms,wind_dir_deg,wind_gust_ms,air_temp_c,baro_mb,waterlevel_verified_m
0,20221001,28.717,47.012,33.406,23.250,3.475,21.054,2.066667,216.458333,3.166667,19.250000,1010.879167,0.955167
1,20221002,37.439,46.926,33.418,23.294,3.454,20.957,2.287500,217.166667,3.050000,19.841667,1012.595833,0.927333
2,20221003,46.564,45.266,33.377,23.694,3.456,19.307,2.279167,235.750000,3.087500,20.229167,1014.079167,0.899667
3,20221004,55.428,43.889,33.317,23.990,3.439,17.962,1.491667,172.500000,2.050000,19.504167,1013.445833,0.903083
4,20221005,62.660,44.128,33.304,23.917,3.427,18.227,1.645833,238.833333,2.045833,18.700000,1013.991667,0.892917


In [47]:
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    # "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    # # "Solidity", "texture_uniformity", "texture_smoothness",
    # # "texture_average_gray_level", "texture_entropy",
    # # "texture_average_contrast", "H90", "H180", "Hflip",
    # # "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")

# Force shared daily index
df = df.asfreq("D")

df_filled = df.copy()
was_imputed = pd.DataFrame(index=df.index)

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)
    was_imputed[col] = df[col].isna()

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    
    df_norm[col] = (df_filled[col] - mu) / sigma

predictors = [col for col in features if col != "Lpoly_expected_ml"]

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)
y_norm = df_mv["Lpoly_expected_ml"].values
df_mv = df_mv[["t", "Lpoly_expected_ml"] + predictors]

In [48]:
df_mv.head()

,t,Lpoly_expected_ml,Area,Biovolume,MajorAxisLength
0,1,0.285287,1.059338,0.967595,0.956625
1,2,0.065656,1.016480,0.917837,0.908034
2,3,-0.128818,0.671753,0.512658,0.637894
3,4,-0.018094,0.379651,0.165389,0.433051
4,5,0.891676,0.756008,0.593485,0.724495


https://sugiharalab.github.io/EDM_Documentation/parameters/

In [50]:
## THIS SHIT IS NOT WORKING WHAT AM I DOING WRONG
N = len(df_mv)
res_mv = pyEDM.Multiview(
    dataFrame=df_mv,
    columns =" ".join(features),
    target="Lpoly_expected_ml",
    E=3,
    tau=1,
    Tp=1,
    lib=f"1 {N}",
    pred=f"1 {N}",
)

obs = res_mv["Observations"].to_numpy()
pred = res_mv["Predictions"].to_numpy()

mask = np.isfinite(obs) & np.isfinite(pred)

rho_mv = np.corrcoef(obs[mask], pred[mask])[0, 1]
print("Multiview rho:", rho_mv)

RuntimeError: Validate() Simplex: target Lpoly_expected_ml(t-0) not found in dataFrame.